# Multi-Indices STAC - Abril 2026Procesa imagenes Sentinel-2 L2A de abril 2026 calculando 5 indices(MSAVI2, S2REP, LAI_RedEdge, Cab_RedEdge, Kc_Actual) con mascara SCLy exportacion por indice a carpetas separadas (PNG + TIF + CSV).**Auto-detecta** si se ejecuta en Google Colab o en PC local.**ROI**: Shapefile (4ParcelasDefinidas.zip) - 4 parcelas**Periodo**: 2026-04-01 a 2026-04-30**Indices**: MSAVI2, S2REP, LAI_RedEdge, Cab_RedEdge, Kc_Actual**Filtro nubes**: mascara SCL (clase 4,5) + etiquetado Despejada/Nublada

In [ ]:
# CELDA 1: AUTO-DETECCION DE ENTORNOimport sys, subprocess, os, importlibEN_COLAB = 'google.colab' in sys.modulesLIBS = ['pystac_client','geopandas','rioxarray','rasterio','odc','xarray','matplotlib']FALTAN = [l for l in LIBS if not importlib.util.find_spec(l.split('.')[0])]if EN_COLAB:    print('Entorno: GOOGLE COLAB')    if FALTAN:        get_ipython().system('pip install pystac-client stackstac rioxarray geopandas rasterio odc-stac -q')else:    print('Entorno: PC LOCAL')    if FALTAN:        print(f'Faltan: {FALTAN}')        subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + FALTAN + ['-q'])    else:        print('Todas las librerias ya instaladas.')print('Entorno listo.')

In [ ]:
# CELDA 2: IMPORTACIONESimport glob, numpy as npimport pandas as pd, geopandas as gpdimport xarray as xr, rioxarray, rasteriofrom datetime import datetimeimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltfrom pystac_client import Clientfrom odc.stac import loadfrom rasterio.features import geometry_maskif EN_COLAB:    from google.colab import driveprint('Importaciones completadas.')

In [ ]:
# CELDA 3: CONFIGURACIONif EN_COLAB:    drive.mount('/content/drive')    SHAPEFILE_PATH = '/content/drive/MyDrive/Tesis/GEE Murcott/ROI/Parcelas+Definidas.zip'    BASE_DIR = os.path.join(os.path.dirname(SHAPEFILE_PATH), 'output', 'Imagenes')else:    SCRIPT_DIR = os.getcwd()    SHAPEFILE_PATH = os.path.join(SCRIPT_DIR, '4ParcelasDefinidas.zip')    if not os.path.exists(SHAPEFILE_PATH):        SHAPEFILE_PATH = './4ParcelasDefinidas.zip'    if not os.path.exists(SHAPEFILE_PATH):        SHAPEFILE_PATH = input('Ruta del shapefile: ').strip()    BASE_DIR = os.path.join(SCRIPT_DIR, 'output', 'Imagenes')PERIODO_INICIO = '2026-04-01'PERIODO_FIN = '2026-04-30'LAI_factor = 0.15Cab_factor = 2kc_slope = 1.15kc_intercept = 0.1UMBRAL_NUBES = 10INDICES_CONFIG = {    'MSAVI2':      {'cmap': 'RdYlGn', 'vmin': 0.2, 'vmax': 0.8},    'S2REP':       {'cmap': 'turbo',  'vmin': 705, 'vmax': 740},    'LAI_RedEdge': {'cmap': 'Greens', 'vmin': 0.1, 'vmax': 6.0},    'Cab_RedEdge': {'cmap': 'YlGn',   'vmin': 0,   'vmax': 100},    'Kc_Actual':   {'cmap': 'BrBG',   'vmin': 0.2, 'vmax': 1.3},}LISTA_INDICES = list(INDICES_CONFIG.keys())os.makedirs(BASE_DIR, exist_ok=True)print('Configuracion cargada.')print(f'  Periodo: {PERIODO_INICIO} a {PERIODO_FIN}')

In [ ]:
# CELDA 4: FUNCIONES AUXILIARESdef cargar_shapefile(ruta):    print('Cargando shapefile...')    gdf = gpd.read_file(ruta)    print(f'  Features: {len(gdf)}')    gdf_geo = gdf if (gdf.crs and gdf.crs.is_geographic) else gdf.to_crs('EPSG:4326')    bbox = gdf_geo.total_bounds    print(f'  BBox: {bbox}')    return gdf_geo, bboxdef conectar_stac():    print('Conectando a Earth Search (AWS)...')    catalog = Client.open('https://earth-search.aws.element84.com/v1')    print('  Conexion exitosa.')    return catalogdef aplicar_mascara_scl(scl, clases_validas=None):    if clases_validas is None:        clases_validas = [4, 5]    mask = xr.zeros_like(scl, dtype=bool)    for c in clases_validas:        mask = mask | (scl == c)    return maskdef aplicar_mascara_geometrica(da, gdf):    x_name = next((n for n in ['x','lon','longitude'] if n in da.coords), None)    y_name = next((n for n in ['y','lat','latitude'] if n in da.coords), None)    if not x_name or not y_name:        return da    transform = da.rio.transform()    ny = da.sizes[y_name]    nx = da.sizes[x_name]    mascara = ~geometry_mask(gdf.geometry.values, transform=transform, out_shape=(ny, nx))    mascara_da = xr.DataArray(mascara, dims=(y_name, x_name),                               coords={y_name: da[y_name], x_name: da[x_name]})    return da.where(mascara_da)def exportar_tif(da, path, crs='EPSG:4326', nodata=-9999):    try:        da2 = da.copy()        x_name = next((n for n in ['x','lon','longitude'] if n in da2.coords), None)        y_name = next((n for n in ['y','lat','latitude'] if n in da2.coords), None)        if x_name and y_name:            da2 = da2.rename({x_name: 'x', y_name: 'y'})        da2.rio.set_spatial_dims('x', 'y', inplace=True)        if not da2.rio.crs:            da2 = da2.rio.write_crs(crs)        da2.rio.to_raster(path, dtype='float32', compress='lzw', nodata=nodata)    except Exception:        data_arr = da.values.astype('float32') if hasattr(da, 'values') else da        ny, nx = data_arr.shape        with rasterio.open(path, 'w', driver='GTiff', height=ny, width=nx,            count=1, dtype='float32', crs=rasterio.crs.CRS.from_string(crs),            compress='lzw', nodata=nodata) as dst:            dst.write(data_arr, 1)def exportar_png(data, path, cmap_name='RdYlGn', vmin=0, vmax=1, dpi=200):    cmap = plt.get_cmap(cmap_name)    cmap.set_bad(color='white', alpha=0)    plt.figure(figsize=(10, 10))    plt.imshow(np.where(np.isnan(np.asarray(data)), np.nan, np.asarray(data)),               cmap=cmap, vmin=vmin, vmax=vmax)    plt.axis('off')    plt.savefig(path, bbox_inches='tight', pad_inches=0, dpi=dpi)    plt.close()def calcular_msavi2(nir, red):    nir_a = np.asarray(nir)    red_a = np.asarray(red)    disc = (2 * nir_a + 1) ** 2 - 8 * (nir_a - red_a)    disc = np.where(disc < 0, np.nan, disc)    msavi2 = (2 * nir_a + 1 - np.sqrt(disc)) / 2    msavi2 = xr.where((nir + red) == 0, np.nan, msavi2)    return msavi2.clip(0, 1)def calcular_s2rep(b4, b5, b6, b7):    denom = b6 - b5    s2rep = xr.where(np.abs(denom) < 1e-4, np.nan,        705 + 35 * (((b4 + b7) / 2 - b5) / denom))    return s2rep.where((s2rep >= 705) & (s2rep <= 740))def calcular_lai(s2rep):    return xr.clip((s2rep - 700) * LAI_factor, 0.1, 6.0)def calcular_cab(s2rep):    return xr.clip((s2rep - 700) * Cab_factor, 0, 100)def calcular_kc(msavi2):    return xr.clip(msavi2 * kc_slope + kc_intercept, 0.2, 1.3)def calcular_estadisticas_parcela(da, gdf, nombre_indice):    stats = []    col_parcela = 'name' if 'name' in gdf.columns else gdf.columns[0]    for i2, row in gdf.iterrows():        m = aplicar_mascara_geometrica(da, gdf.iloc[[i2]])        v = m.values.flatten()        v = v[~np.isnan(v)]        stats.append({'ID_Parcela': row.get(col_parcela, str(i2)),                       f'{nombre_indice}_mean': float(np.mean(v)) if len(v) > 0 else '',                       f'{nombre_indice}_std': float(np.std(v)) if len(v) > 0 else '',                       'pixeles_validos': int(len(v))})    return statsprint('Funciones auxiliares cargadas.')

In [ ]:
# CELDA 5: CARGA SHAPEFILE + BUSQUEDA STACgdf_geo, bbox = cargar_shapefile(SHAPEFILE_PATH)catalog = conectar_stac()search = catalog.search(    collections=['sentinel-2-l2a'],    bbox=list(bbox),    datetime=f'{PERIODO_INICIO}/{PERIODO_FIN}',)items = list(search.items())if len(items) == 0:    print('No se encontraron imagenes. Verificar cobertura.')print(f'Total escenas encontradas: {len(items)}')for item in items:    ts = item.datetime    fecha = ts.strftime('%Y-%m-%d') if ts else 'N/A'    hora = ts.strftime('%H:%M:%S') if ts else 'N/A'    cloud = item.properties.get('eo:cloud_cover', -1)    estado = 'Despejada' if 0 <= cloud < UMBRAL_NUBES else ('Nublada' if cloud >= 0 else 'Sin dato')    print(f'  {fecha} {hora} | nubes={cloud:.0f}% | {estado}')

In [ ]:
# CELDA 6: PROCESAMIENTO DE ESCENASstats_por_indice = {i: [] for i in LISTA_INDICES}label_mes = 'Abril2026'contador = 0; exp = 0; omit = 0for idx, item in enumerate(items):    ts = item.datetime    if ts is None: continue    fecha = ts.strftime('%Y%m%d')    hora = ts.strftime('%H%M%S')    cv = item.properties.get('eo:cloud_cover', -1)    est = 'Despejada' if cv < UMBRAL_NUBES and cv >= 0 else 'Nublada'    print(f'[{idx+1}/{len(items)}] {fecha} {hora} | {est}')    # Verificar los 5 TIFs antes de decidir si procesar la escena    tifs_existentes = 0; tifs_detalle = {}    for nom in LISTA_INDICES:        base_nom = f'{nom}_Abril_{contador + 1:03d}_{fecha}_{hora}_{est}'        tif_path = os.path.join(BASE_DIR, nom, label_mes, 'TIF', f'{base_nom}.tif')        existe = os.path.exists(tif_path)        tifs_detalle[nom] = {'existe': existe, 'base': base_nom, 'tif_path': tif_path}        if existe: tifs_existentes += 1    if tifs_existentes == len(LISTA_INDICES):        print('  Todos los TIFs ya existen. Saltando.'); omit += 1; continue    elif tifs_existentes > 0:        print(f'  {tifs_existentes}/{len(LISTA_INDICES)} TIFs ya existen. Procesando solo faltantes.')    try:        ds = load([item], bands=['red','rededge1','rededge2','rededge3','nir','scl'],                  bbox=list(bbox), crs='EPSG:4326', resolution=0.0001, groupby=None)        if ds.sizes.get('time', 0) == 0: continue        esc = ds.isel(time=0)        b4 = esc['red'].astype('float32') / 10000        b5 = esc['rededge1'].astype('float32') / 10000        b6 = esc['rededge2'].astype('float32') / 10000        b7 = esc['rededge3'].astype('float32') / 10000        b8 = esc['nir'].astype('float32') / 10000        msc = aplicar_mascara_scl(esc['scl'])        b4 = b4.where(msc); b5 = b5.where(msc)        b6 = b6.where(msc); b7 = b7.where(msc); b8 = b8.where(msc)        b4 = aplicar_mascara_geometrica(b4, gdf_geo)        b5 = aplicar_mascara_geometrica(b5, gdf_geo)        b6 = aplicar_mascara_geometrica(b6, gdf_geo)        b7 = aplicar_mascara_geometrica(b7, gdf_geo)        b8 = aplicar_mascara_geometrica(b8, gdf_geo)        msavi2 = calcular_msavi2(b8, b4)        s2rep = calcular_s2rep(b4, b5, b6, b7)        lai = calcular_lai(s2rep)        cab = calcular_cab(s2rep)        kc = calcular_kc(msavi2)        inds = {'MSAVI2':msavi2,'S2REP':s2rep,'LAI_RedEdge':lai,                'Cab_RedEdge':cab,'Kc_Actual':kc}        contador += 1        for nom, da in inds.items():            vals = np.asarray(da.values) if hasattr(da, 'values') else np.asarray(da)            sf = 'SinDatos' if np.all(np.isnan(vals)) else est            arch = f'{nom}_Abril_{contador:03d}_{fecha}_{hora}_{sf}'            d = os.path.join(BASE_DIR, nom, label_mes)            os.makedirs(f'{d}/TIF', exist_ok=True)            os.makedirs(f'{d}/PNG', exist_ok=True)            tif_p = f'{d}/TIF/{arch}.tif'            if os.path.exists(tif_p): continue            exportar_tif(da, tif_p)            cfg = INDICES_CONFIG[nom]            exportar_png(vals, f'{d}/PNG/{arch}.png',                         cmap_name=cfg['cmap'], vmin=cfg['vmin'], vmax=cfg['vmax'])            stats_parcelas = calcular_estadisticas_parcela(da, gdf_geo, nom)            for st in stats_parcelas:                stats_por_indice[nom].append({'fecha':fecha,'hora':hora,                    'id_escena':arch,'nubes_porciento':cv,'estado_nubosidad':sf,                    **st, 'ruta_tif':tif_p, 'ruta_png':f'{d}/PNG/{arch}.png'})            exp += 1        del ds    except Exception as e:        print(f'  ERROR: {e}')        continue

In [ ]:
# CELDA 7: GUARDAR CSV POR INDICEfor nom in LISTA_INDICES:    reg = stats_por_indice[nom]    if not reg: print(f'{nom}: sin registros'); continue    df = pd.DataFrame(reg)    csv_p = os.path.join(BASE_DIR, nom, label_mes, 'estadisticas.csv')    df.to_csv(csv_p, index=False, encoding='utf-8')    print(f'{nom}: {len(df)} registros')

In [ ]:
# CELDA 8: REPORTE FINALprint('='*60)print('   REPORTE FINAL')print('='*60)print(f'  Escenas: {len(items)} | Exportados: {exp} | Omitidos: {omit}')print()tt = 0; tp = 0for nom in LISTA_INDICES:    d = os.path.join(BASE_DIR, nom, label_mes)    nt = len(glob.glob(f'{d}/TIF/*.tif')) if os.path.isdir(f'{d}/TIF') else 0    np = len(glob.glob(f'{d}/PNG/*.png')) if os.path.isdir(f'{d}/PNG') else 0    csv_ok = 'CSV' if os.path.exists(f'{d}/estadisticas.csv') else '---'    tt += nt; tp += np    print(f'  {nom:15s}: {nt:3d} TIFs | {np:3d} PNGs | {csv_ok}')print()print(f'  TOTAL: {tt} TIFs + {tp} PNGs')print(f'  Directorio: {BASE_DIR}')print('='*60)print('  Procesamiento completado!')print('='*60)